# Fresh preparation, report labels and split audit
CPU. Attach ONLY the original RSNA competition. No previous pseudo labels, folds, slot plans or checkpoints.
This fast audit checks series directories, not every DICOM file. The report rules are a transparent preliminary baseline, not validated clinical labels. Review label_audit.csv before training. Save & Run All; attach this output to notebooks 02 and 03.

In [ ]:
from pathlib import Path
import json, time, hashlib
import numpy as np
import pandas as pd
ROOTS = ['/kaggle/input/datasets/aymenslimani', '/kaggle/input']
def locate(name):
    hits = sorted(set(p.resolve() for root in ROOTS if Path(root).exists() for p in Path(root).rglob(name)))
    if not hits: raise FileNotFoundError(f'Attach the saved previous notebook output containing {name}')
    if len(hits)>1 and len({hashlib.sha256(p.read_bytes()).hexdigest() for p in hits})>1:
        raise ValueError(f'Conflicting inputs; select one attachment: {hits}')
    return hits[0]
TARGETS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']

In [ ]:
import re
from sklearn.model_selection import GroupKFold
COMP = next((Path(p) for p in ['/kaggle/input/competitions/rsna-knee-abnormality-detection','/kaggle/input/rsna-knee-abnormality-detection'] if (Path(p)/'train.csv').exists()), None)
if COMP is None: raise FileNotFoundError('Add the RSNA competition input')
OUT = Path('/kaggle/working/rsna_fresh_preparation'); OUT.mkdir(exist_ok=True)
N_FOLDS = 5
UID = 'StudyInstanceUID'
train = pd.read_csv(COMP/'train.csv', dtype=str)
series = pd.read_csv(COMP/'train_series.csv', dtype=str)
print('Report columns:', train.columns.tolist())
print('Series columns:', series.columns.tolist())
REPORT_COLUMN = next((c for c in ['Report','report','ReportText','report_text'] if c in train), None)
if REPORT_COLUMN is None: raise ValueError('Set REPORT_COLUMN to the report-text column printed above')
if UID not in train or UID not in series or 'SeriesInstanceUID' not in series: raise ValueError('Missing study/series identifier columns')
if train[UID].isna().any() or train[UID].duplicated().any(): raise ValueError('Missing/duplicate report study identifiers')
if series[[UID,'SeriesInstanceUID']].isna().any().any(): raise ValueError('Missing series identifiers')
image_root = COMP/'train_series'
if not image_root.exists(): raise FileNotFoundError(image_root)
series['series_path'] = [str(image_root/u/s) for u,s in zip(series[UID],series.SeriesInstanceUID)]
t0=time.time()
series['folder_exists'] = [Path(p).is_dir() for p in series.series_path]
series.to_csv(OUT/'series_audit.csv',index=False)
print('Directory audit seconds:', round(time.time()-t0,1), 'missing:', int((~series.folder_exists).sum()))
print('Studies without series:', len(set(train[UID])-set(series.loc[series.folder_exists,UID])))
print('Empty reports:', int(train[REPORT_COLUMN].fillna('').str.strip().eq('').sum()))

Labels use explicit abnormality mentions and conservative local negation/uncertainty handling. Unmentioned targets receive zero weight, rather than being assumed negative. Negation scope is only a heuristic; contradictory mentions are masked and exported for review. Rules and report column are saved for reproducibility.

In [ ]:
RULES = {
 'ACL': r'(?:acl|anterior cruciate ligament).{0,65}(?:tear|ruptur|disrupt|sprain)|(?:tear|ruptur).{0,40}(?:acl|anterior cruciate)',
 'MCL': r'(?:mcl|medial collateral ligament).{0,65}(?:tear|sprain|injur|disrupt)',
 'Medial Meniscus': r'medial menisc.{0,65}(?:tear|extrus|macerat)|tear.{0,40}medial menisc',
 'Lateral Meniscus': r'lateral menisc.{0,65}(?:tear|extrus|macerat)|tear.{0,40}lateral menisc',
 'Medial OA': r'medial (?:compartment|tibiofemoral).{0,80}(?:chondr|cartilage loss|osteoarth|degenerat)',
 'Lateral OA': r'lateral (?:compartment|tibiofemoral).{0,80}(?:chondr|cartilage loss|osteoarth|degenerat)',
 'PF OA': r'(?:patellofemoral|patellar|trochlear).{0,80}(?:chondr|cartilage loss|osteoarth|degenerat)',
 'Effusion': r'(?:joint )?effusion', 'Synovitis': r'synovitis|synovial (?:thicken|hypertroph)',
 "Baker's": r'baker.?s? cyst|popliteal cyst', 'Contusion': r'contusion|bone bruise',
 'Fracture': r'fracture',
}
NEG = r'\b(?:no|without|absent|negative for|no evidence of)\b'
UNCERTAIN = r'\b(?:possible|possibly|may|might|questionable|suspect|cannot exclude)\b'
def label_report(text):
    sentences = re.split(r'[.;\n]+', str(text).lower())
    ys, ws, states = [], [], []
    for target in TARGETS:
        evidence=[]
        for sentence in sentences:
            for match in re.finditer(RULES[target], sentence):
                context=sentence[max(0,match.start()-55):match.end()]
                evidence.append('uncertain' if re.search(UNCERTAIN,context) else 'negative' if re.search(NEG,context) else 'positive')
        kinds=set(evidence)
        state='unmentioned' if not kinds else 'conflict' if len(kinds)>1 else next(iter(kinds))
        ys.append(float(state=='positive')); ws.append(float(state in ['positive','negative'])); states.append(state)
    return ys,ws,states
rows=[]; audit=[]
for _,r in train.iterrows():
    y,w,states=label_report(r[REPORT_COLUMN] if pd.notna(r[REPORT_COLUMN]) else '')
    rows.append([r[UID]]+y+w)
    audit.append([r[UID],r[REPORT_COLUMN]]+states)
manifest=pd.DataFrame(rows,columns=[UID]+['y_'+str(i) for i in range(12)]+['w_'+str(i) for i in range(12)])
# Prefer patient identifiers if available. Otherwise group identical normalized reports.
# This fallback cannot prove that different reports belong to different patients.
patient_col=next((c for c in ['PatientID','patient_id','PatientId'] if c in train),None)
normalized=train[REPORT_COLUMN].fillna('').str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
manifest['group']=[hashlib.sha256(s.encode()).hexdigest() if s else 'empty_'+u for s,u in zip(normalized,train[UID])]
if patient_col:
    if train[patient_col].isna().any(): raise ValueError('Patient identifiers missing; resolve before freezing splits')
    manifest['group']=train[patient_col].values
manifest=manifest[manifest[UID].isin(series.loc[series.folder_exists,UID])].sort_values(UID).reset_index(drop=True)
manifest['fold']=-1
for fold,(_,val) in enumerate(GroupKFold(N_FOLDS).split(manifest,groups=manifest.group)):
    manifest.loc[val,'fold']=fold
assert manifest.groupby('group').fold.nunique().max()==1
manifest.to_csv(OUT/'fresh_manifest.csv',index=False)
pd.DataFrame(audit,columns=[UID,'report']+TARGETS).to_csv(OUT/'label_audit.csv',index=False)
coverage=pd.DataFrame({'target':TARGETS,'positive':manifest[['y_'+str(i) for i in range(12)]].sum().values,'supervised':manifest[['w_'+str(i) for i in range(12)]].sum().values})
display(coverage); display(manifest.groupby('fold').size())
meta={'targets':TARGETS,'rules':RULES,'report_column':REPORT_COLUMN,'group_method':patient_col or 'normalized_report_hash_fallback','manifest_sha256':hashlib.sha256((OUT/'fresh_manifest.csv').read_bytes()).hexdigest(),'n_folds':N_FOLDS,'scope':'heuristic report supervision; audit required'}
(OUT/'fresh_preparation.json').write_text(json.dumps(meta,indent=2))
print('Saved:',OUT, 'Review label_audit.csv; weak coverage requires improving rules before larger training.')